# 基于图割模型的交互式图像分割实验：学生练习版

本实验实现一个基于 **图割模型（Graph Cut）** 的交互式图像分割程序。练习版保留网络图像读取、鼠标拖动标定、结果显示、最小割调用和参数对比等框架，将图割建模中最关键、最需要理解的部分设置为 `TODO`，需要学生补全。

本练习重点补全：

1. `compute_beta`：根据相邻像素颜色差异估计平滑项参数 `beta`。
2. `build_graph_cut_graph`：构建包含源点、汇点、像素节点、终端边和邻接边的 s-t 图。

## 1. 图割模型相关知识

图割分割是一种经典的交互式图像分割方法。它通常要求用户提供少量前景种子点和背景种子点，然后算法自动判断其他未标注像素更应该属于前景还是背景。

### 1.1 图像到图模型的转换

对于一幅图像，可以把每个像素看成图中的一个节点。除此之外，还添加两个特殊节点：

1. `S`：源点，表示前景。
2. `T`：汇点，表示背景。

图中的边分为两类：

1. **终端边（t-link）**：连接像素节点与 `S`、`T`，表示该像素属于前景或背景的代价。
2. **邻接边（n-link）**：连接相邻像素节点，表示相邻像素被分到不同类别时的平滑代价。

### 1.2 能量函数

二值图割分割可以写成能量最小化问题：

$$
E(L)=\sum_p D_p(L_p)+\lambda \sum_{(p,q)\in N} V_{pq}[L_p \ne L_q]
$$

其中：

1. `p` 表示像素。
2. `L_p` 表示像素 `p` 的类别，前景或背景。
3. `D_p(L_p)` 是数据项，表示像素 `p` 属于某一类别的代价。
4. `V_{pq}` 是平滑项，表示相邻像素 `p` 和 `q` 被分到不同类别时的代价。
5. `lambda` 控制分割结果的平滑程度。

### 1.3 最小割思想

构造好图之后，寻找一条代价最小的割，将图分成包含 `S` 的部分和包含 `T` 的部分：

1. 与 `S` 连通的像素被判为前景。
2. 与 `T` 连通的像素被判为背景。

本实验使用 `networkx.minimum_cut` 求解 s-t 最小割。为了让课堂运行速度更稳定，网络图像会被缩小到较小尺寸后再构图。

## 2. 图割分割计算步骤

完整流程如下：

1. 从网络读取图像，并转换为 RGB 图像。
2. 将图像缩放到适合图割计算的尺寸。
3. 通过鼠标拖动刷选前景种子点和背景种子点。
4. 根据前景种子点计算前景颜色均值。
5. 根据背景种子点计算背景颜色均值。
6. 对每个像素计算数据项：
   - 越接近前景颜色，属于前景的代价越小。
   - 越接近背景颜色，属于背景的代价越小。
7. 对每对相邻像素计算平滑项：
   - 两个像素颜色越相似，被分开的代价越大。
   - 两个像素颜色差异越大，被分开的代价越小。
8. 构建包含 `S`、`T`、像素节点、终端边和邻接边的图。
9. 调用最小割算法得到前景区域和背景区域。
10. 显示分割掩膜、边界和前景提取结果。

## 学生练习任务与代码补全步骤

本实验难度相对较高。建议按照下面顺序补全，不要直接从构图代码中间开始写。

### 1. 先理解已有函数

1. `pixel_node(y, x)`：把像素坐标转换成图节点名称。
2. `compute_color_models(image, foreground_points, background_points)`：根据前景和背景种子点计算颜色均值。
3. `run_graph_cut_segmentation(...)`：调用你补全的构图函数，然后使用 `networkx.minimum_cut` 求最小割。
4. 后面的显示函数只负责可视化，不是本次考查重点。

### 2. 补全 `compute_beta`

`beta` 用于控制相邻像素颜色差异对平滑边权的影响。实现步骤：

1. 将 `image` 转换为 `float64`，避免颜色差值计算溢出。
2. 计算水平方向相邻像素差值：
   - `image[:, 1:, :] - image[:, :-1, :]`
3. 计算垂直方向相邻像素差值：
   - `image[1:, :, :] - image[:-1, :, :]`
4. 对每个颜色差向量计算平方距离：
   - `R`、`G`、`B` 三个通道平方后求和。
5. 将所有平方距离合并，计算平均平方距离 `mean_sq_diff`。
6. 如果 `mean_sq_diff` 很小，说明图像几乎没有颜色变化，返回 `0.0`。
7. 否则返回：

$$
\beta = \frac{1}{2 \cdot mean\_sq\_diff}
$$

### 3. 补全 `build_graph_cut_graph`

构图是本实验最核心的部分。建议分成三段完成。

第一段：初始化图模型

1. 获取图像高度 `h` 和宽度 `w`。
2. 将图像转换为 `float64`。
3. 调用 `compute_color_models` 得到前景颜色均值和背景颜色均值。
4. 调用 `compute_beta` 得到平滑项参数。
5. 创建前景种子点集合和背景种子点集合。
6. 创建 `nx.DiGraph()`。
7. 添加源点 `S` 和汇点 `T`。

第二段：添加终端边 t-link

1. 遍历每一个像素 `(y, x)`。
2. 用 `pixel_node(y, x)` 得到当前像素节点。
3. 计算当前像素到前景均值的颜色距离，得到 `d_fg`。
4. 计算当前像素到背景均值的颜色距离，得到 `d_bg`。
5. 如果当前像素是前景种子点：
   - 添加 `source -> node` 的大权重边。
   - 添加 `node -> sink` 的 0 权重边。
6. 如果当前像素是背景种子点：
   - 添加 `source -> node` 的 0 权重边。
   - 添加 `node -> sink` 的大权重边。
7. 如果当前像素不是种子点：
   - 添加 `source -> node`，容量为 `d_bg`。
   - 添加 `node -> sink`，容量为 `d_fg`。

第三段：添加邻接边 n-link

1. 再次遍历每个像素。
2. 只考虑右邻居和下邻居，避免重复添加。
3. 计算相邻像素颜色差平方和 `sq_diff`。
4. 根据公式计算平滑边权：

$$
weight = \lambda \cdot e^{-\beta \cdot sq\_diff}
$$

5. 因为使用的是有向图，需要同时添加两个方向：
   - `node -> neighbor`
   - `neighbor -> node`
6. 最后返回 `graph, source, sink, fg_mean, bg_mean`。

### 4. 补全后的检查顺序

1. 先运行到“确认种子点”单元，确保 `foreground_points` 和 `background_points` 不为空。
2. 补全并运行 `compute_beta`，可以先打印 `beta` 看是否为非负数。
3. 补全并运行 `build_graph_cut_graph`。
4. 运行“使用最小割完成图像分割”单元。
5. 运行结果显示单元，观察分割掩膜和边界。
6. 最后调整 `lambda_smooth`，观察平滑项权重对分割结果的影响。

## 3. 导入基础库

In [ ]:
from io import BytesIO

try:
    # JupyterLab / Notebook 中推荐使用 widget 后端，鼠标拖动交互更稳定。
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    try:
        get_ipython().run_line_magic("matplotlib", "notebook")
    except Exception:
        pass

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import requests
from PIL import Image

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 4. 读取网络图像

本实验默认使用一张网络图像。若课堂网络环境暂时无法访问该地址，可以替换 `IMAGE_URL` 为其他可访问的图片地址。

In [ ]:
IMAGE_URLS = [
    "https://img95.699pic.com/photo/60063/3888.jpg_wh860.jpg"
]


def load_rgb_image_from_url(url):
    """
    从网络地址读取 RGB 图像。

    参数：
        url: 网络图像地址

    返回：
        image: uint8 类型 RGB 图像，形状为 H × W × 3
    """
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "Mozilla/5.0"},
        verify=False,
    )
    response.raise_for_status()
    image = Image.open(BytesIO(response.content)).convert("RGB")
    return np.array(image, dtype=np.uint8)


def load_first_available_image(urls):
    """
    依次尝试多个网络图像地址，返回第一张成功读取的图像。
    """
    errors = []
    for url in urls:
        try:
            image = load_rgb_image_from_url(url)
            return image, url
        except Exception as exc:
            errors.append(f"{url} -> {exc}")
    raise RuntimeError("所有网络图像读取失败：\n" + "\n".join(errors))


def resize_image_keep_ratio(image, max_size=80):
    """
    按比例缩小图像，避免图割构图过大导致运行缓慢。

    参数：
        image: 输入 RGB 图像
        max_size: 缩放后长边最大长度

    返回：
        resized_image: 缩放后的 RGB 图像
    """
    h, w = image.shape[:2]
    scale = min(max_size / max(h, w), 1.0)
    new_w = int(round(w * scale))
    new_h = int(round(h * scale))
    pil_image = Image.fromarray(image)
    resized = pil_image.resize((new_w, new_h), Image.BILINEAR)
    return np.array(resized, dtype=np.uint8)


original_image, used_image_url = load_first_available_image(IMAGE_URLS)

image = resize_image_keep_ratio(original_image, max_size=80)
print("网络图像读取成功：", used_image_url)
print("原始图像尺寸：", original_image.shape)
print("图割计算图像尺寸：", image.shape)

plt.figure(figsize=(6, 5))
plt.imshow(image)
plt.title("用于图割分割的网络图像")
plt.axis("off")
plt.show()

## 5. 拖动鼠标标定前景和背景种子点

运行下面单元后，在图像上拖动鼠标刷选种子点：

1. 按住鼠标左键拖动：添加前景种子点。
2. 按住鼠标右键拖动：添加背景种子点。
3. 按键盘 `c`：清空所有种子点，重新标定。
4. 标定完成后，继续运行后面的单元。

如果拖动没有反应，请先确认已经运行导入库单元中的交互后端设置。JupyterLab 推荐使用：

```python
%matplotlib widget
```

如果提示缺少组件，可以在环境中安装：

```python
pip install ipympl
```

如果当前环境仍然不支持鼠标交互，可以直接使用下一单元提供的默认种子点，或手动修改 `foreground_points` 与 `background_points`。

In [ ]:
class SeedSelector:
    """
    交互式种子点标定器。

    左键拖动刷前景种子点，右键拖动刷背景种子点。
    """

    def __init__(self, image, brush_radius=2):
        self.image = image
        self.brush_radius = brush_radius
        self.foreground_points = set()
        self.background_points = set()
        self.active_button = None

        self.fig, self.ax = plt.subplots(figsize=(7, 6))
        self.ax.imshow(image)
        self.ax.set_title("左键拖动：前景；右键拖动：背景；按 c 清空")
        self.ax.axis("off")

        self.foreground_artist = self.ax.scatter([], [], c="lime", s=18, marker="s", alpha=0.75, label="前景种子")
        self.background_artist = self.ax.scatter([], [], c="red", s=18, marker="s", alpha=0.75, label="背景种子")
        self.ax.legend(loc="upper right")

        self.cid_press = self.fig.canvas.mpl_connect("button_press_event", self.on_press)
        self.cid_motion = self.fig.canvas.mpl_connect("motion_notify_event", self.on_motion)
        self.cid_release = self.fig.canvas.mpl_connect("button_release_event", self.on_release)
        self.cid_key = self.fig.canvas.mpl_connect("key_press_event", self.on_key)

    def add_seed_disk(self, y, x, target):
        """
        在鼠标位置附近加入一个小圆盘区域的种子点。

        参数：
            y, x: 鼠标所在像素坐标
            target: "foreground" 或 "background"
        """
        h, w = self.image.shape[:2]
        r = self.brush_radius

        for yy in range(y - r, y + r + 1):
            for xx in range(x - r, x + r + 1):
                if yy < 0 or yy >= h or xx < 0 or xx >= w:
                    continue
                if (yy - y) ** 2 + (xx - x) ** 2 > r ** 2:
                    continue

                point = (yy, xx)
                if target == "foreground":
                    self.foreground_points.add(point)
                    self.background_points.discard(point)
                elif target == "background":
                    self.background_points.add(point)
                    self.foreground_points.discard(point)

    def update_artists(self):
        """
        刷新图像上的种子点显示。
        """
        if self.foreground_points:
            fg_y, fg_x = zip(*sorted(self.foreground_points))
            self.foreground_artist.set_offsets(np.column_stack([fg_x, fg_y]))
        else:
            self.foreground_artist.set_offsets(np.empty((0, 2)))

        if self.background_points:
            bg_y, bg_x = zip(*sorted(self.background_points))
            self.background_artist.set_offsets(np.column_stack([bg_x, bg_y]))
        else:
            self.background_artist.set_offsets(np.empty((0, 2)))

        self.fig.canvas.draw_idle()

    def add_from_event(self, event):
        """
        根据鼠标事件添加种子点。
        """
        if event.inaxes != self.ax or event.xdata is None or event.ydata is None:
            return

        x = int(round(event.xdata))
        y = int(round(event.ydata))

        if self.active_button == 1:
            self.add_seed_disk(y, x, "foreground")
        elif self.active_button == 3:
            self.add_seed_disk(y, x, "background")
        else:
            return

        self.update_artists()

    def on_press(self, event):
        """
        鼠标按下时记录当前按钮，并立即添加一次种子点。
        """
        if event.button in (1, 3):
            self.active_button = event.button
            self.add_from_event(event)

    def on_motion(self, event):
        """
        鼠标拖动时持续添加种子点。
        """
        if self.active_button in (1, 3):
            self.add_from_event(event)

    def on_release(self, event):
        """
        鼠标释放时停止刷选。
        """
        self.active_button = None

    def on_key(self, event):
        """
        按 c 清空全部种子点。
        """
        if event.key == "c":
            self.foreground_points.clear()
            self.background_points.clear()
            self.update_artists()
            print("已清空前景和背景种子点")

    def get_points(self):
        """
        返回前景和背景种子点列表，点格式为 (y, x)。
        """
        return sorted(self.foreground_points), sorted(self.background_points)


selector = SeedSelector(image, brush_radius=2)
plt.show()

## 6. 确认种子点

运行下面单元读取鼠标拖动标定得到的种子点。如果没有成功进行鼠标交互，程序会使用默认种子点。也可以在本单元中直接手动修改 `foreground_points` 和 `background_points`。

注意：种子点格式为 `(y, x)`，不是 `(x, y)`。

In [ ]:
# 如果上一个单元的点击交互可用，运行本单元会读取点击得到的种子点。
# 如果没有进行点击，程序会使用下面这组默认种子点。

h, w = image.shape[:2]
default_foreground_points = [
    (int(h * 0.52), int(w * 0.50)),
    (int(h * 0.60), int(w * 0.46)),
    (int(h * 0.66), int(w * 0.55)),
]
default_background_points = [
    (int(h * 0.08), int(w * 0.08)),
    (int(h * 0.12), int(w * 0.88)),
    (int(h * 0.90), int(w * 0.12)),
    (int(h * 0.90), int(w * 0.88)),
]

if "selector" in globals():
    clicked_fg, clicked_bg = selector.get_points()
else:
    clicked_fg, clicked_bg = [], []

foreground_points = clicked_fg if len(clicked_fg) > 0 else default_foreground_points
background_points = clicked_bg if len(clicked_bg) > 0 else default_background_points

print("前景种子点：", foreground_points)
print("背景种子点：", background_points)

plt.figure(figsize=(7, 6))
plt.imshow(image)
if foreground_points:
    fy, fx = zip(*foreground_points)
    plt.scatter(fx, fy, c="lime", s=60, marker="o", edgecolors="black", label="前景种子")
if background_points:
    by, bx = zip(*background_points)
    plt.scatter(bx, by, c="red", s=60, marker="x", label="背景种子")
plt.title("前景与背景种子点")
plt.axis("off")
plt.legend()
plt.show()

## 7. 构建图割模型

In [ ]:
def pixel_node(y, x):
    """
    将像素坐标转换为图节点编号。
    """
    return (y, x)


def compute_color_models(image, foreground_points, background_points):
    """
    根据前景和背景种子点计算颜色均值。

    参数：
        image: RGB 图像
        foreground_points: 前景种子点列表，每个点为 (y, x)
        background_points: 背景种子点列表，每个点为 (y, x)

    返回：
        fg_mean: 前景颜色均值
        bg_mean: 背景颜色均值
    """
    if len(foreground_points) == 0 or len(background_points) == 0:
        raise ValueError("至少需要一个前景种子点和一个背景种子点")

    fg_pixels = np.array([image[y, x] for y, x in foreground_points], dtype=np.float64)
    bg_pixels = np.array([image[y, x] for y, x in background_points], dtype=np.float64)
    fg_mean = fg_pixels.mean(axis=0)
    bg_mean = bg_pixels.mean(axis=0)
    return fg_mean, bg_mean


def compute_beta(image):
    """
    TODO：根据相邻像素颜色差异估计平滑项中的 beta。

    beta 用于控制颜色差异对邻接边权重的影响：
        weight = lambda_smooth * exp(-beta * sq_diff)

    参数：
        image: RGB 图像，形状为 H × W × 3

    返回：
        beta: 平滑项参数
    """
    # TODO 1：将 image 转换为 np.float64。
    # TODO 2：计算水平方向相邻像素差值。
    # TODO 3：计算垂直方向相邻像素差值。
    # TODO 4：分别计算每个差值向量的 RGB 平方距离。
    # TODO 5：合并所有平方距离，并计算平均平方距离 mean_sq_diff。
    # TODO 6：如果 mean_sq_diff <= 1e-12，返回 0.0。
    # TODO 7：否则返回 1.0 / (2.0 * mean_sq_diff)。
    raise NotImplementedError("请补全 compute_beta 函数")


def build_graph_cut_graph(
    image,
    foreground_points,
    background_points,
    lambda_smooth=35.0,
    seed_weight=1e6,
):
    """
    TODO：构建图割所需的 s-t 图。

    参数：
        image: RGB 图像
        foreground_points: 前景种子点列表
        background_points: 背景种子点列表
        lambda_smooth: 平滑项权重，越大分割越平滑
        seed_weight: 种子点强约束权重

    返回：
        graph: networkx 有向图
        source: 源点名称
        sink: 汇点名称
        fg_mean: 前景颜色均值
        bg_mean: 背景颜色均值
    """
    # TODO 1：获取图像高度 h 和宽度 w。
    # TODO 2：将 image 转换为 np.float64，保存为 image_float。
    # TODO 3：调用 compute_color_models 得到 fg_mean 和 bg_mean。
    # TODO 4：调用 compute_beta 得到 beta。
    # TODO 5：将 foreground_points 和 background_points 转换为集合，便于快速判断。
    # TODO 6：创建 nx.DiGraph，并设置 source = "S"，sink = "T"。
    # TODO 7：向图中添加 source 和 sink 两个节点。
    # TODO 8：设置 color_scale = 255.0 ** 2，用于归一化颜色距离。

    # TODO 9：遍历每个像素，为像素节点添加终端边 t-link。
    #   9.1：node = pixel_node(y, x)
    #   9.2：color = image_float[y, x]
    #   9.3：d_fg = 像素颜色到 fg_mean 的平方距离 / color_scale
    #   9.4：d_bg = 像素颜色到 bg_mean 的平方距离 / color_scale
    #   9.5：如果当前点是前景种子点，强制连到 source。
    #   9.6：如果当前点是背景种子点，强制连到 sink。
    #   9.7：普通像素添加 source -> node 和 node -> sink 两条终端边。

    # TODO 10：再次遍历每个像素，为右邻居和下邻居添加邻接边 n-link。
    #   10.1：找到当前像素的右邻居和下邻居。
    #   10.2：计算当前像素与邻居像素颜色差平方和 sq_diff。
    #   10.3：weight = lambda_smooth * np.exp(-beta * sq_diff)
    #   10.4：因为 graph 是有向图，需要添加 node -> neighbor 和 neighbor -> node。

    # TODO 11：返回 graph, source, sink, fg_mean, bg_mean。
    raise NotImplementedError("请补全 build_graph_cut_graph 函数")

## 8. 使用最小割完成图像分割

In [ ]:
def run_graph_cut_segmentation(
    image,
    foreground_points,
    background_points,
    lambda_smooth=35.0,
):
    """
    运行图割分割。

    参数：
        image: RGB 图像
        foreground_points: 前景种子点列表
        background_points: 背景种子点列表
        lambda_smooth: 平滑项权重

    返回：
        mask: 前景掩膜，前景为 True
        cut_value: 最小割代价
        fg_mean: 前景颜色均值
        bg_mean: 背景颜色均值
    """
    graph, source, sink, fg_mean, bg_mean = build_graph_cut_graph(
        image,
        foreground_points,
        background_points,
        lambda_smooth=lambda_smooth,
    )

    cut_value, partition = nx.minimum_cut(graph, source, sink, capacity="capacity")
    reachable, non_reachable = partition

    h, w = image.shape[:2]
    mask = np.zeros((h, w), dtype=bool)
    for y in range(h):
        for x in range(w):
            if pixel_node(y, x) in reachable:
                mask[y, x] = True

    return mask, cut_value, fg_mean, bg_mean


lambda_smooth = 35.0
mask, cut_value, fg_mean, bg_mean = run_graph_cut_segmentation(
    image,
    foreground_points,
    background_points,
    lambda_smooth=lambda_smooth,
)

print("最小割代价：", cut_value)
print("前景颜色均值：", fg_mean)
print("背景颜色均值：", bg_mean)

## 9. 显示图割分割结果

In [ ]:
def compute_mask_boundary(mask):
    """
    根据二值掩膜计算边界像素。
    """
    boundary = np.zeros_like(mask, dtype=bool)
    boundary[:, 1:] |= mask[:, 1:] != mask[:, :-1]
    boundary[1:, :] |= mask[1:, :] != mask[:-1, :]
    return boundary


boundary = compute_mask_boundary(mask)
foreground_result = image.copy()
foreground_result[~mask] = 255

overlay = image.copy()
overlay[boundary] = np.array([255, 0, 0], dtype=np.uint8)

plt.figure(figsize=(14, 8))

plt.subplot(2, 3, 1)
plt.imshow(image)
plt.title("原图")
plt.axis("off")

plt.subplot(2, 3, 2)
plt.imshow(image)
if foreground_points:
    fy, fx = zip(*foreground_points)
    plt.scatter(fx, fy, c="lime", s=45, marker="o", edgecolors="black", label="前景")
if background_points:
    by, bx = zip(*background_points)
    plt.scatter(bx, by, c="red", s=45, marker="x", label="背景")
plt.title("用户种子点")
plt.axis("off")
plt.legend()

plt.subplot(2, 3, 3)
plt.imshow(mask, cmap="gray")
plt.title("图割前景掩膜")
plt.axis("off")

plt.subplot(2, 3, 4)
plt.imshow(overlay)
plt.title("分割边界")
plt.axis("off")

plt.subplot(2, 3, 5)
plt.imshow(foreground_result)
plt.title("前景提取结果")
plt.axis("off")

plt.subplot(2, 3, 6)
plt.imshow(image)
plt.imshow(mask, cmap="Greens", alpha=0.35)
plt.title("前景区域叠加")
plt.axis("off")

plt.tight_layout()
plt.show()

## 10. 观察平滑项权重对结果的影响

In [ ]:
lambda_values = [5.0, 20.0, 50.0]

plt.figure(figsize=(12, 4))
for i, lam in enumerate(lambda_values):
    mask_lam, cut_lam, _, _ = run_graph_cut_segmentation(
        image,
        foreground_points,
        background_points,
        lambda_smooth=lam,
    )

    plt.subplot(1, len(lambda_values), i + 1)
    plt.imshow(mask_lam, cmap="gray")
    plt.title(f"lambda={lam}\ncut={cut_lam:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 11. 实验小结

本实验实现了基于图割模型的交互式图像分割。

需要掌握的重点：

1. 图割模型将每个像素看成图中的节点。
2. 源点 `S` 表示前景，汇点 `T` 表示背景。
3. 终端边表示像素属于前景或背景的代价。
4. 邻接边表示相邻像素被分到不同类别时的平滑代价。
5. 用户点击的前景和背景种子点会对对应像素施加强约束。
6. 最小割会把图分成源点侧和汇点侧，从而得到前景和背景分割。
7. `lambda_smooth` 越大，结果越平滑，但可能损失细节；越小，结果更依赖颜色模型，可能产生噪声。

思考题：

1. 如果前景和背景颜色非常接近，图割分割结果会受到什么影响？
2. 如果用户选错了种子点，分割结果会怎样变化？
3. 除了颜色均值，还可以用哪些方法建立前景和背景的数据项？
4. 图割方法与区域生长方法有什么区别？